[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/e8/e8.ipynb)

# E8 Reflections

This example computes the root system of E8, the 240 vectors of equal length in eight dimensions on which the largest exceptional Lie group is built, and the familiar picture of them as eight concentric rings of thirty. Reflecting in the hyperplane square to any root permutes all the roots. Eight such reflections generate them all, and their product is a rotation that returns to where it started after thirty steps, turning four perpendicular planes at once.

The eight mirrors are vectors, and their product is a rotor, `coxeter = reduce(mul, mirrors)`, whose action on vectors, `coxeter >> Vector`, is the whole motion as one map. Applied thirty times to one root from each of eight orbits, it reaches all 240. Averaged with its adjoint, `(action + action.adjoint()) / 2`, the motion keeps each of its invariant planes and scales it by the cosine of its turn, so one symmetric eigensolve finds the four planes and how fast each turns: 1, 7, 11 and 13 times in thirty steps. The slowest is the Coxeter plane, where the roots fall on eight rings.

In the notation of Lie theory the Coxeter element of E8 has order $h = 30$ and eigenvalues $e^{2\pi i m / h}$ for the exponents $m = 1, 7, 11, 13, 17, 19, 23, 29$, which pair into the four plane rotations by $m$ and $h - m$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import reduce
from operator import mul

import numpy as np
from IPython.display import Image, display

from numga import Algebra, NumpyContext, stack
from examples.animation import save_animation, save_figure

from examples.math.e8 import render

np.set_printoptions(precision=6, suppress=True)

In [ ]:
ga = Algebra("a+b+c+d+e+f+g+h+")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
VectorMap = ga.gatype((Vector, Vector))

## 1. Root orbits

Eight mirror normals, called simple roots, generate all the reflections. Their product is the Coxeter rotor: its action returns to the identity after 30 steps. Eight starting roots trace disjoint orbits, each containing 30 roots.

In [ ]:
n_steps = 30
simple_roots = stack([                                                          # Vector
    (mv.a + mv.h - mv.b - mv.c - mv.d - mv.e - mv.f - mv.g) / 2,
    mv.a + mv.b,
    mv.b - mv.a,
    mv.c - mv.b,
    mv.d - mv.c,
    mv.e - mv.d,
    mv.f - mv.e,
    mv.g - mv.f,
])                                                                           # [mirrors] Vector

In [ ]:
def orbit(action: VectorMap, seeds: Vector, n_steps: int) -> Iterator[Vector]:
    """Successive images of a batch of seeds under one motion."""
    for _ in range(n_steps):
        yield seeds                                                           # [orbits] Vector
        seeds = action(seeds)                                                 # [orbits] Vector


mirrors = simple_roots.normalized()                                            # [mirrors] Vector
coxeter = reduce(mul, mirrors)                                               # [] Rotor
# Acting on the vector type gives the motion as a unary extensor.
action = coxeter >> Vector                                                   # [] Vector <- Vector


def orbit_starts(simple_roots: Vector) -> Iterator[Vector]:
    """One starting root for each disjoint orbit of the Coxeter motion."""
    preceding_reflections = Vector                                           # [] Vector <- Vector
    for root in simple_roots:
        # The first root stays unchanged; each later root is acted on by the preceding reflections.
        yield preceding_reflections(root)                                    # [] Vector
        # Compose on the right to preserve the order of the Coxeter product.
        preceding_reflections = preceding_reflections(root.normalized() >> Vector)# [] Vector <- Vector


starting_roots = stack(orbit_starts(simple_roots))                             # [orbits] Vector
roots = stack(orbit(action, starting_roots, n_steps))                           # [steps, orbits] Vector

## 2. The Coxeter plane

Forward and backward rotation have opposite sideways components. Their average therefore scales each invariant plane by the cosine of its turn angle. The paired eigendirections recover four perpendicular planes; the slowest is the Coxeter plane, where the eight orbits appear as eight rings. Colours identify the orbits, and faint edges connect nearest neighbours in eight dimensions.

In [ ]:
symmetric = (action + action.adjoint()) / 2                                     # [] Vector <- Vector
cosines, directions = symmetric.eigh()                                        # [directions] Scalar, Vector

# Each cosine occurs twice. Descending cosines order the planes from slow to fast.
plane_x = directions[::-2]                                                    # [planes] Vector
# The sideways part of the motion fixes the positive direction in each plane.
plane_y = (action(plane_x) - cosines[::-2] * plane_x).normalized()               # [planes] Vector
planes = plane_x ^ plane_y                                                    # [planes] Bivector
rates = cosines[::-2].arccos() * n_steps / (2 * np.pi)                          # [planes] Scalar

print("Turns in 30 steps:", render.winding_rates(rates))
display(Image(filename=str(save_figure(
    render.draw_roots(roots, plane_x[0], plane_y[0]), "e8_roots",
))))

## 3. Four simultaneous turns

Each coloured dot follows one root, viewed in all four planes at once. From left to right it makes 1, 7, 11 and 13 revolutions before the whole motion repeats. Faint dots mark the discrete roots; the continuous paths pass between them. Trails cover the same duration in every panel, so their angular spans show the different rotation rates. This cell takes about ten seconds, mostly drawing the animation.

In [ ]:
n_substeps = 6
n_frames = n_steps * n_substeps
duration_ms = 60

In [ ]:
# The four perpendicular plane generators commute, so their turns combine by addition.
generator = -(planes * rates).sum(axis=0) * np.pi / n_steps                    # [] Bivector
# Repeated small rotations recover each Coxeter step.
advance = (generator / n_substeps).exp() >> Vector                             # [] Vector <- Vector
turning = orbit(advance, starting_roots, n_frames)

display(Image(filename=str(save_animation(
    render.animate_planes(turning, roots, plane_x, plane_y), "e8_planes", duration_ms,
))))

The rings are the shadows of root orbits under a common eight-dimensional motion. The four plane bivectors supply its commuting generators, while the motion as a map on vectors exposes those planes through one eigensolve. The Clifford construction and the relation between these rotation angles and E8's Coxeter exponents are developed in Pierre-Philippe Dechant's [The E8 Geometry from a Clifford Perspective](https://eprints.whiterose.ac.uk/id/eprint/96267/1/E8Geometry.pdf).